# Seasonal Agriculture Performance Analysis

**Major Project — VOIS AICTE Batch 2026–2027**

This notebook analyzes the provided agricultural dataset to investigate seasonal differences, patterns, relationships, variations, and potential insights for seasonal agricultural planning.

### Project objectives
- Explore and understand the dataset.
- Clean and prepare the data.
- Examine agricultural performance across seasons.
- Identify seasonal patterns and trends.
- Investigate relationships between conditions, resources, and outcomes.
- Compare relevant groups across seasons.
- Detect unusual observations and variations.
- Apply statistical and visualization techniques.
- Develop evidence-based conclusions and recommendations.


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 2. Load the Dataset

In [ ]:
# Update this path if the notebook is moved to another folder.
FILE_PATH = "seasonal_agriculture_performance_dataset.csv"

data = pd.read_csv(FILE_PATH)

print("Dataset shape:", data.shape)
display(data.head())


## 3. Dataset Structure and Data Types

In [ ]:
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])

display(pd.DataFrame({
    "column": data.columns,
    "dtype": data.dtypes.astype(str).values,
    "missing_values": data.isna().sum().values,
    "unique_values": [data[c].nunique(dropna=True) for c in data.columns]
}))


## 4. Data Cleaning and Preparation

In [ ]:
# Standardize column names
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Remove duplicate records
before = len(data)
data = data.drop_duplicates().copy()
after = len(data)

print(f"Duplicate rows removed: {before - after}")

# Convert columns that are predominantly numeric
for col in data.columns:
    if data[col].dtype == "object":
        cleaned = (
            data[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        numeric = pd.to_numeric(cleaned, errors="coerce")
        non_null = data[col].notna().sum()
        if non_null and numeric.notna().sum() / non_null >= 0.80:
            data[col] = numeric

display(data.head())


## 5. Missing Values

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(data) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct
})

display(missing_table[missing_table["missing_count"] > 0])

# Conservative treatment:
# Numeric columns -> median imputation
# Categorical columns -> mode imputation
for col in data.columns:
    if data[col].isna().sum() == 0:
        continue

    if pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].fillna(data[col].median())
    else:
        mode = data[col].mode(dropna=True)
        if len(mode):
            data[col] = data[col].fillna(mode.iloc[0])

print("Remaining missing values:", int(data.isna().sum().sum()))


## 6. Descriptive Statistics

In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns

if len(numeric_cols):
    display(data[numeric_cols].describe().T)
else:
    print("No numeric columns were detected.")


## 7. Identify Seasonal Variable

In [ ]:
season_candidates = [
    c for c in data.columns
    if "season" in c or "period" in c
]

print("Possible seasonal columns:", season_candidates)

# Select the first likely seasonal column.
# Change SEASON_COL manually if another column is more appropriate.
SEASON_COL = season_candidates[0] if season_candidates else None

if SEASON_COL:
    print("Selected seasonal column:", SEASON_COL)
    display(data[SEASON_COL].value_counts(dropna=False))
else:
    print("No obvious season/period column was automatically detected.")


## 8. Explore Categorical Variables

In [ ]:
categorical_cols = data.select_dtypes(exclude=np.number).columns

for col in categorical_cols:
    print(f"\n### {col}")
    print("Unique values:", data[col].nunique())
    display(data[col].value_counts(dropna=False).head(15))


## 9. Seasonal Performance Analysis

In [ ]:
# Select a numeric outcome column.
# The automatic selection favors columns containing yield, production,
# output, profit, income, or performance.
target_candidates = [
    c for c in numeric_cols
    if any(k in c for k in ["yield", "production", "output", "profit", "income", "performance"])
]

print("Possible performance columns:", target_candidates)

TARGET_COL = target_candidates[0] if target_candidates else None

if SEASON_COL and TARGET_COL:
    seasonal_summary = (
        data.groupby(SEASON_COL)[TARGET_COL]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("mean", ascending=False)
    )
    display(seasonal_summary)

    plt.figure(figsize=(9, 5))
    data.boxplot(column=TARGET_COL, by=SEASON_COL)
    plt.title(f"{TARGET_COL.title()} Distribution by Season")
    plt.suptitle("")
    plt.xlabel("Season")
    plt.ylabel(TARGET_COL.title())
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("Set SEASON_COL and TARGET_COL after reviewing the dataset.")


## 10. Compare Numeric Variables Across Seasons

In [ ]:
if SEASON_COL:
    seasonal_means = data.groupby(SEASON_COL)[numeric_cols].mean()
    display(seasonal_means)

    # Plot the first few numeric variables to keep the analysis readable.
    plot_cols = list(numeric_cols[:6])

    for col in plot_cols:
        plt.figure(figsize=(9, 5))
        data.groupby(SEASON_COL)[col].mean().plot(kind="bar")
        plt.title(f"Average {col.title()} by Season")
        plt.xlabel("Season")
        plt.ylabel(f"Mean {col.title()}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print("Seasonal column not available.")


## 11. Correlation Analysis

In [ ]:
if len(numeric_cols) >= 2:
    corr = data[numeric_cols].corr(numeric_only=True)

    plt.figure(figsize=(10, 7))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation Matrix of Numeric Variables")
    plt.tight_layout()
    plt.show()

    display(corr.round(3))
else:
    print("At least two numeric columns are required.")


## 12. Relationship Between Performance and Other Numeric Factors

In [ ]:
if TARGET_COL:
    other_numeric = [c for c in numeric_cols if c != TARGET_COL]

    for col in other_numeric[:8]:
        plt.figure(figsize=(8, 5))
        plt.scatter(data[col], data[TARGET_COL], alpha=0.6)
        plt.xlabel(col.title())
        plt.ylabel(TARGET_COL.title())
        plt.title(f"{TARGET_COL.title()} vs {col.title()}")
        plt.tight_layout()
        plt.show()

        valid = data[[col, TARGET_COL]].dropna()
        if len(valid) >= 3:
            r, p = stats.pearsonr(valid[col], valid[TARGET_COL])
            print(f"{col}: Pearson r = {r:.3f}, p-value = {p:.4g}")


## 13. Statistical Comparison of Seasons

In [ ]:
if SEASON_COL and TARGET_COL:
    groups = [
        group[TARGET_COL].dropna().values
        for _, group in data.groupby(SEASON_COL)
    ]
    group_names = list(data.groupby(SEASON_COL).groups.keys())

    if len(groups) == 2:
        statistic, p_value = stats.ttest_ind(groups[0], groups[1], equal_var=False)
        print("Welch's t-test")
        print("Groups:", group_names)
        print("Statistic:", round(statistic, 4))
        print("p-value:", p_value)
    elif len(groups) > 2:
        statistic, p_value = stats.f_oneway(*groups)
        print("One-way ANOVA")
        print("Seasons:", group_names)
        print("F-statistic:", round(statistic, 4))
        print("p-value:", p_value)
        print("\nInterpretation: A p-value below 0.05 suggests that at least one seasonal mean differs statistically from the others.")
    else:
        print("Not enough seasonal groups for a statistical comparison.")
else:
    print("Set SEASON_COL and TARGET_COL first.")


## 14. Outlier Detection

In [ ]:
outlier_summary = []

for col in numeric_cols:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((data[col] < lower) | (data[col] > upper)).sum()

    outlier_summary.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(count),
        "outlier_percent": round(count / len(data) * 100, 2)
    })

display(pd.DataFrame(outlier_summary).sort_values("outlier_count", ascending=False))


## 15. Seasonal Trends / Group Comparisons

In [ ]:
if SEASON_COL:
    for col in numeric_cols:
        summary = data.groupby(SEASON_COL)[col].mean().sort_values(ascending=False)

        print(f"\n{col}:")
        display(summary.to_frame("mean"))

        plt.figure(figsize=(9, 5))
        plt.plot(summary.index.astype(str), summary.values, marker="o")
        plt.title(f"Seasonal Mean Trend — {col.title()}")
        plt.xlabel("Season")
        plt.ylabel(f"Mean {col.title()}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()


## 16. Key Findings Template

In [ ]:
# This section prints dataset-specific values that can be used when writing conclusions.

if SEASON_COL:
    season_counts = data[SEASON_COL].value_counts()
    print("Number of observations by season:")
    display(season_counts)

if SEASON_COL and TARGET_COL:
    means = data.groupby(SEASON_COL)[TARGET_COL].mean().sort_values(ascending=False)
    best_season = means.index[0]
    worst_season = means.index[-1]

    print(f"Highest average {TARGET_COL}: {best_season} ({means.iloc[0]:,.3f})")
    print(f"Lowest average {TARGET_COL}: {worst_season} ({means.iloc[-1]:,.3f})")
    print(f"Difference: {(means.iloc[0] - means.iloc[-1]):,.3f}")

print("\nUse the tables, plots, correlations, statistical tests, and outlier analysis above to write the final evidence-based findings.")


## 17. Conclusions

The final conclusions should be based only on the results produced above. The analysis should address:

1. How agricultural performance varies across seasons.
2. Which seasonal patterns are strongest.
3. Which characteristics change between seasons.
4. Differences in resource usage and environmental conditions.
5. Relationships between relevant agricultural factors and performance.
6. Whether observed seasonal differences are statistically meaningful.
7. Any unusual or unexpected patterns.
8. What conclusions are reasonably supported by the available data.


## 18. Data-Driven Recommendations

Recommendations should be tied directly to observed evidence. Suitable areas include:

- Planning agricultural activities according to seasonal performance.
- Improving resource allocation in seasons with higher resource requirements.
- Investigating environmental conditions associated with lower performance.
- Replicating practices associated with stronger seasonal outcomes where appropriate.
- Monitoring unusual observations and potential outliers.
- Collecting additional data where the current dataset cannot establish causation.
- Using future seasonal observations to validate whether identified patterns remain consistent.


## 19. Final Project Checklist

- [x] Dataset loaded
- [x] Data structure inspected
- [x] Data cleaning performed
- [x] Missing values examined and handled
- [x] Descriptive statistics generated
- [x] Seasonal variable identified
- [x] Performance comparison performed
- [x] Visualizations created
- [x] Correlation analysis performed
- [x] Statistical comparison included
- [x] Outlier analysis included
- [x] Findings/conclusion framework included
- [x] Recommendations framework included

**Important:** Review `SEASON_COL` and `TARGET_COL` after opening the notebook. If the dataset contains a more appropriate season or performance field than the automatically selected one, replace those two variables before running the complete analysis.
